# RASP-SFOD — Compact Export + Final Evaluation


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DRIVE_PROJECT='/content/drive/MyDrive/MedRT-SFOD'
LOCAL_PROJECT='/content/MedRT-SFOD'
RUNS='/content/drive/MyDrive/MedRT-SFOD-runs'
!rm -rf $LOCAL_PROJECT
!mkdir -p $LOCAL_PROJECT
!rsync -a --exclude dataset --exclude runs '$DRIVE_PROJECT/' '$LOCAL_PROJECT/'
%cd $LOCAL_PROJECT
!pip -q install -e .
!pip -q install -r requirements-rasp.txt


In [ ]:
OUT_DIR=f'{RUNS}/c2f_rasp'
STATE=f'{OUT_DIR}/checkpoints/rasp_training_state_epoch_60.pt'
LATENT=f'{OUT_DIR}/checkpoints/yolo26_rasp_latent_epoch_60.pt'
COMPACT=f'{OUT_DIR}/yolo26m_rasp_compact.pt'
TARGET_YAML=f'{DRIVE_PROJECT}/dataset/c2f_yolo/foggy_cityscapes.yaml'


## Physical compaction + numerical equivalence


In [ ]:
!PYTHONPATH='$PWD' python scripts/YOLO26/export_rasp_compact.py \
  --state '$STATE' --latent_model '$LATENT' --out '$COMPACT' \
  --device 0 --verify --count_macs


## Final target evaluation (labels used only now, not for model selection)


In [ ]:
!PYTHONPATH='$PWD' python scripts/YOLO26/eval_rasp_student.py \
  --model '$COMPACT' --data '$TARGET_YAML' --imgsz 1024 --batch 4 --device 0
